In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [2]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### WDPA: World Database on Protected Areas (polygons)

In [3]:
pa = ee.FeatureCollection("WCMC/WDPA/current/polygons")

# Rasterize protected area polygons
pa_raster = ee.Image().byte().paint(pa, 1)

# distance to nearest protected area in meters
distance_to_pa = (
    pa_raster
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(30)  # adjust if your dataset resolution differs
    .rename("dist_pa_m")
    .clip(panama_geom)
)


Map.addLayer(panama_geom, {}, 'Panama Boundary')
Map.addLayer(pa, {}, 'Protected Areas')
Map.addLayer(distance_to_pa, {}, 'Distance to Protected Areas (m)')

In [4]:
# Display map
Map.centerObject(panama_geom, 7)
Map

Map(center=[8.515838945899919, -80.10966640141515], controls=(WidgetControl(options=['position', 'transparent_…